In [1]:
# -*- coding: utf-8 -*-
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Any
import itertools

# Kaggle 路径配置
WORK_DIR = "/kaggle/working"

DATA_PATH = '/kaggle/input/datasets/nilesh789/eurosat-rgb/2750'

print(f"数据集路径: {DATA_PATH}")

SAVE_DIR = os.path.join(WORK_DIR, "checkpoints")
FIGURE_DIR = os.path.join(WORK_DIR, "figures")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

数据集路径: /kaggle/input/datasets/nilesh789/eurosat-rgb/2750


In [2]:
#数据加载与预处理 
class DataLoader:
    #加载 EuroSAT 数据集，划分训练/验证/测试集，并提供 mini-batch 迭代器
    def __init__(self, data_path: str, val_ratio: float = 0.1, test_ratio: float = 0.2, batch_size: int = 64):
        self.data_path = data_path
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.batch_size = batch_size
        self.classes = None
        self.class_to_idx = None
        self.X_train, self.y_train = None, None
        self.X_val, self.y_val = None, None
        self.X_test, self.y_test = None, None

    def load_data(self) -> None:
        #读取图像和标签，划分数据集并归一化
        class_folders = [d for d in os.listdir(self.data_path) if os.path.isdir(os.path.join(self.data_path, d))]
        self.classes = sorted(class_folders)
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        print(f"找到类别: {self.classes}")

        images, labels = [], []
        for cls in self.classes:
            folder = os.path.join(self.data_path, cls)
            for file in os.listdir(folder):
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    img_path = os.path.join(folder, file)
                    img = Image.open(img_path).convert('RGB')
                    img_array = np.array(img, dtype=np.float32) / 255.0
                    images.append(img_array)
                    labels.append(self.class_to_idx[cls])

        X = np.array(images)
        y = np.array(labels)
        #展平图像: (H, W, C) -> (H*W*C)
        X = X.reshape(X.shape[0], -1)
        print(f"数据集大小: {X.shape[0]} 样本, 输入维度: {X.shape[1]}")

        #分层划分
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=self.test_ratio, stratify=y, random_state=RANDOM_SEED)
        val_ratio_adj = self.val_ratio / (1 - self.test_ratio)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_ratio_adj, stratify=y_temp, random_state=RANDOM_SEED)

        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test
        print(f"训练集: {len(X_train)}, 验证集: {len(X_val)}, 测试集: {len(X_test)}")

    def get_batches(self, X: np.ndarray, y: np.ndarray, shuffle: bool = True):
        #生成 mini-batch 迭代器
        n_samples = X.shape[0]
        indices = np.arange(n_samples)
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n_samples, self.batch_size):
            end = min(start + self.batch_size, n_samples)
            batch_idx = indices[start:end]
            yield X[batch_idx], y[batch_idx]

In [3]:
#激活函数及其导数 
class Activation:
    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)

    @staticmethod
    def relu_derivative(x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

    @staticmethod
    def sigmoid(x: np.ndarray) -> np.ndarray:
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def sigmoid_derivative(x: np.ndarray) -> np.ndarray:
        sig = Activation.sigmoid(x)
        return sig * (1 - sig)

    @staticmethod
    def tanh(x: np.ndarray) -> np.ndarray:
        return np.tanh(x)

    @staticmethod
    def tanh_derivative(x: np.ndarray) -> np.ndarray:
        return 1 - np.tanh(x) ** 2

In [4]:
#全连接层 
class LinearLayer:
    def __init__(self, in_features: int, out_features: int):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.b = np.zeros((1, out_features))
        self.grad_W = None
        self.grad_b = None
        self.input = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.input = x
        return x @ self.W + self.b

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        self.grad_W = self.input.T @ grad_output
        self.grad_b = np.sum(grad_output, axis=0, keepdims=True)
        grad_input = grad_output @ self.W.T
        return grad_input

    def update(self, lr: float, weight_decay: float = 0.0):
        self.W -= lr * (self.grad_W + weight_decay * self.W)
        self.b -= lr * self.grad_b

In [5]:
#神经网络模型
class NeuralNetwork:
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int, activation: str = 'relu'):
        self.fc1 = LinearLayer(input_dim, hidden_dim)
        self.fc2 = LinearLayer(hidden_dim, num_classes)
        self.activation_name = activation
        self.activation_func = getattr(Activation, activation)
        self.activation_derivative = getattr(Activation, f"{activation}_derivative")
        self.cache_activation_input = None

    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        z1 = self.fc1.forward(x)
        self.cache_activation_input = z1
        a1 = self.activation_func(z1)
        logits = self.fc2.forward(a1)
        return logits

    def backward(self, grad_output: np.ndarray) -> None:
        grad_a1 = self.fc2.backward(grad_output)
        grad_z1 = grad_a1 * self.activation_derivative(self.cache_activation_input)
        self.fc1.backward(grad_z1)

    def update(self, lr: float, weight_decay: float = 0.0):
        self.fc1.update(lr, weight_decay)
        self.fc2.update(lr, weight_decay)

    def compute_l2_loss(self, weight_decay: float) -> float:
        l2_loss = 0.5 * weight_decay * (np.sum(self.fc1.W ** 2) + np.sum(self.fc2.W ** 2))
        return l2_loss

    def save_weights(self, filepath: str):
        weights = {
            'fc1_W': self.fc1.W, 'fc1_b': self.fc1.b,
            'fc2_W': self.fc2.W, 'fc2_b': self.fc2.b,
            'activation': self.activation_name,
            'hidden_dim': self.fc1.W.shape[1]
        }
        with open(filepath, 'wb') as f:
            pickle.dump(weights, f)

    def load_weights(self, filepath: str):
        with open(filepath, 'rb') as f:
            weights = pickle.load(f)
        self.fc1.W = weights['fc1_W']
        self.fc1.b = weights['fc1_b']
        self.fc2.W = weights['fc2_W']
        self.fc2.b = weights['fc2_b']
        self.activation_name = weights.get('activation', 'relu')
        self.activation_func = getattr(Activation, self.activation_name)
        self.activation_derivative = getattr(Activation, f"{self.activation_name}_derivative")

In [6]:
#损失函数 
class CrossEntropyLoss:
    @staticmethod
    def forward(logits: np.ndarray, targets: np.ndarray) -> float:
        shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
        exp_logits = np.exp(shifted_logits)
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        n_samples = logits.shape[0]
        correct_log_probs = -np.log(probs[np.arange(n_samples), targets] + 1e-8)
        loss = np.mean(correct_log_probs)
        return loss

    @staticmethod
    def backward(logits: np.ndarray, targets: np.ndarray) -> np.ndarray:
        shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
        exp_logits = np.exp(shifted_logits)
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        n_samples = logits.shape[0]
        grad = probs.copy()
        grad[np.arange(n_samples), targets] -= 1
        grad /= n_samples
        return grad

In [7]:
#训练器 
class Trainer:
    def __init__(self, model: NeuralNetwork, lr: float, weight_decay: float, lr_decay: float = 0.95, lr_decay_epochs: int = 5):
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.lr_decay = lr_decay
        self.lr_decay_epochs = lr_decay_epochs
        self.loss_fn = CrossEntropyLoss()
        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []

    def train_epoch(self, data_loader: DataLoader) -> float:
        epoch_loss = 0.0
        n_batches = 0
        for X_batch, y_batch in data_loader.get_batches(data_loader.X_train, data_loader.y_train, shuffle=True):
            logits = self.model.forward(X_batch)
            loss = self.loss_fn.forward(logits, y_batch)
            l2_loss = self.model.compute_l2_loss(self.weight_decay)
            total_loss = loss + l2_loss
            epoch_loss += total_loss
            n_batches += 1
            grad = self.loss_fn.backward(logits, y_batch)
            self.model.backward(grad)
            self.model.update(self.lr, self.weight_decay)
        return epoch_loss / n_batches

    def validate(self, data_loader: DataLoader) -> Tuple[float, float]:
        X_val, y_val = data_loader.X_val, data_loader.y_val
        logits = self.model.forward(X_val, training=False)
        loss = self.loss_fn.forward(logits, y_val)
        l2_loss = self.model.compute_l2_loss(self.weight_decay)
        total_loss = loss + l2_loss
        preds = np.argmax(logits, axis=1)
        acc = np.mean(preds == y_val)
        return total_loss, acc

    def train(self, data_loader: DataLoader, epochs: int, patience: int = 10):
        best_val_acc = 0.0
        best_model_path = os.path.join(SAVE_DIR, "best_model.pkl")
        patience_counter = 0

        for epoch in range(1, epochs + 1):
            train_loss = self.train_epoch(data_loader)
            val_loss, val_acc = self.validate(data_loader)
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)

            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                self.model.save_weights(best_model_path)
                patience_counter = 0
                print(f"  -> 保存最佳模型 (准确率: {val_acc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"早停于 epoch {epoch}")
                    break

            if epoch % self.lr_decay_epochs == 0:
                self.lr *= self.lr_decay
                print(f"  学习率衰减至: {self.lr:.6f}")

        self.model.load_weights(best_model_path)
        print(f"训练完成，最佳验证准确率: {best_val_acc:.4f}")
        return best_val_acc

In [8]:
#超参数搜索
class HyperparameterSearch:
    def __init__(self, data_loader: DataLoader, input_dim: int, num_classes: int):
        self.data_loader = data_loader
        self.input_dim = input_dim
        self.num_classes = num_classes

    def grid_search(self, param_grid: Dict[str, List[Any]], epochs: int = 10):
        best_score = 0.0
        best_params = {}
        results = []

        keys = list(param_grid.keys())
        for values in itertools.product(*param_grid.values()):
            params = dict(zip(keys, values))
            print(f"\n测试超参数: {params}")
            model = NeuralNetwork(self.input_dim, params['hidden_dim'], self.num_classes, activation=params['activation'])
            trainer = Trainer(model, lr=params['lr'], weight_decay=params['weight_decay'], lr_decay=0.95, lr_decay_epochs=5)
            val_acc = trainer.train(self.data_loader, epochs=epochs, patience=5)
            results.append({**params, 'val_acc': val_acc})
            if val_acc > best_score:
                best_score = val_acc
                best_params = params

        print("\n下面为超参数搜索结果：")
        for res in sorted(results, key=lambda x: x['val_acc'], reverse=True):
            print(f"{res}")
        print(f"\n最佳超参数: {best_params}, 验证准确率: {best_score:.4f}")
        return best_params, results

In [9]:
#可视化工具
def plot_curves(train_losses, val_losses, val_accuracies, save_path: str):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss Curves')
    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_accuracies, label='Val Accuracy', color='green')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Validation Accuracy')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"曲线图已保存至 {save_path}")

def visualize_first_layer_weights(model: NeuralNetwork, save_path: str, img_shape: Tuple[int, int, int] = (64, 64, 3)):
    W1 = model.fc1.W
    hidden_dim = W1.shape[1]
    norms = np.linalg.norm(W1, axis=0)
    top_indices = np.argsort(norms)[-16:][::-1]
    n_cols = 4
    n_rows = 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))
    for idx, ax in enumerate(axes.flat):
        if idx < len(top_indices):
            filt = W1[:, top_indices[idx]].reshape(img_shape)
            filt_min, filt_max = filt.min(), filt.max()
            if filt_max - filt_min > 1e-8:
                filt_disp = (filt - filt_min) / (filt_max - filt_min)
            else:
                filt_disp = filt
            ax.imshow(filt_disp)
            ax.set_title(f'Filter {top_indices[idx]}\nNorm={norms[top_indices[idx]]:.2f}')
        ax.axis('off')
    plt.suptitle('First Layer Filters (Top 16 by L2 Norm)', fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"第一层权重可视化已保存至 {save_path}")

def error_analysis(model: NeuralNetwork, data_loader: DataLoader, num_examples: int = 6, save_path: str = None):
    X_test, y_test = data_loader.X_test, data_loader.y_test
    logits = model.forward(X_test, training=False)
    preds = np.argmax(logits, axis=1)
    errors = np.where(preds != y_test)[0]
    if len(errors) == 0:
        print("测试集上没有错误样本！")
        return
    selected = np.random.choice(errors, min(num_examples, len(errors)), replace=False)

    classes = data_loader.classes
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(selected):
        img = X_test[idx].reshape(64, 64, 3)
        true_label = classes[y_test[idx]]
        pred_label = classes[preds[idx]]
        ax = plt.subplot(2, 3, i+1)
        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label}")
        ax.axis('off')
    plt.suptitle("Error Analysis: Misclassified Examples", fontsize=14)
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()
    print(f"错例分析图已保存至 {save_path}")

def final_evaluation(model: NeuralNetwork, data_loader: DataLoader):
    X_test, y_test = data_loader.X_test, data_loader.y_test
    logits = model.forward(X_test, training=False)
    preds = np.argmax(logits, axis=1)
    acc = np.mean(preds == y_test)
    print(f"\n最终测试结果如下：")
    print(f"测试集准确率: {acc:.4f}")
    cm = confusion_matrix(y_test, preds)
    print("混淆矩阵:")
    print(cm)
    print("\n分类报告:")
    print(classification_report(y_test, preds, target_names=data_loader.classes))
    return acc, cm

In [10]:
#主流程
def main():
    print("1.加载数据")
    data_loader = DataLoader(DATA_PATH, val_ratio=0.1, test_ratio=0.2, batch_size=64)
    data_loader.load_data()
    input_dim = data_loader.X_train.shape[1]
    num_classes = len(data_loader.classes)
    print(f"输入维度: {input_dim}, 类别数: {num_classes}")

    print("\n2.超参数搜索 (网格搜索) ")
    param_grid = {
        'lr': [1e-3, 5e-4],
        'hidden_dim': [128, 256],
        'weight_decay': [0.0, 1e-4],
        'activation': ['relu', 'sigmoid']
    }
    searcher = HyperparameterSearch(data_loader, input_dim, num_classes)
    best_params, _ = searcher.grid_search(param_grid, epochs=16)

    print("\n3. 使用最佳超参数训练最终模型")
    final_model = NeuralNetwork(input_dim, best_params['hidden_dim'], num_classes, activation=best_params['activation'])
    trainer = Trainer(final_model, lr=best_params['lr'], weight_decay=best_params['weight_decay'],
                      lr_decay=0.95, lr_decay_epochs=10)
    trainer.train(data_loader, epochs=50, patience=12)

    print("\n4. 绘制训练曲线")
    plot_curves(trainer.train_losses, trainer.val_losses, trainer.val_accuracies,
                os.path.join(FIGURE_DIR, "training_curves.png"))

    print("\n5. 第一层权重可视化")
    visualize_first_layer_weights(final_model, os.path.join(FIGURE_DIR, "first_layer_filters.png"))

    print("\n6. 测试集评估")
    final_evaluation(final_model, data_loader)

    print("\n7. 错例分析")
    error_analysis(final_model, data_loader, num_examples=6, save_path=os.path.join(FIGURE_DIR, "error_analysis.png"))

    print("\n所有任务完成！输出文件保存在:", WORK_DIR)

if __name__ == "__main__":
    main()
    

1.加载数据
找到类别: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
数据集大小: 27000 样本, 输入维度: 12288
训练集: 18900, 验证集: 2700, 测试集: 5400
输入维度: 12288, 类别数: 10

2.超参数搜索 (网格搜索) 

测试超参数: {'lr': 0.001, 'hidden_dim': 128, 'weight_decay': 0.0, 'activation': 'relu'}
Epoch   1/16 | Train Loss: 2.1309 | Val Loss: 2.0341 | Val Acc: 0.2637
  -> 保存最佳模型 (准确率: 0.2637)
Epoch   2/16 | Train Loss: 1.9696 | Val Loss: 1.9411 | Val Acc: 0.2674
  -> 保存最佳模型 (准确率: 0.2674)
Epoch   3/16 | Train Loss: 1.8826 | Val Loss: 1.8574 | Val Acc: 0.2893
  -> 保存最佳模型 (准确率: 0.2893)
Epoch   4/16 | Train Loss: 1.8276 | Val Loss: 1.8198 | Val Acc: 0.2870
Epoch   5/16 | Train Loss: 1.7879 | Val Loss: 1.7779 | Val Acc: 0.3804
  -> 保存最佳模型 (准确率: 0.3804)
  学习率衰减至: 0.000950
Epoch   6/16 | Train Loss: 1.7596 | Val Loss: 1.7577 | Val Acc: 0.3456
Epoch   7/16 | Train Loss: 1.7383 | Val Loss: 1.7334 | Val Acc: 0.3722
Epoch   8/16 | Train Loss: 1.7205 | Val Loss: 